# ZuCo validated clock recovery — CPU preparation
Select **CPU**, then **Run all**, after your current TeCo run finishes. Wait for **CPU PREPARATION COMPLETE** before switching runtimes.

The independent audit approved **34 previously rejected blocks**, using 25,401 word-fixation waveform checks (all checked errors below 2 ms). This reconstruction can therefore attempt **88/96 blocks**. Three timing cases remain unresolved; five still lack required eye-event sources. Actual accepted sentiment-trial counts are determined by extraction and matching, not forced to match the paper.

**This must refit each reader's shared cleaning model and matching with the additional blocks.** It cannot simply append new cleaned trials to the old averages. Completed readers in this NEW reconstruction are saved and skipped on resume. An unfinished reader may repeat after an interruption. Exact native-file checksums are checked against the timing evidence before a recovered clock can be used.

**Additional storage: up to 4 GB**, including about 1 GB temporary permutation files and a safety reserve. CPU trial files have approximately 2.7 GB allowance. All accepted raw/clean trials retain 105 channels ×201 samples, float32. No extra copy of the large source EEG is retained. Free Drive quota is checked before extraction.

New destination: `Thesis/Results/word_sentiment_eeg/zuco_paper_figures_recovered/`. Your completed `zuco_paper_figures_staged/` results are preserved. This is a new reconstruction and does not silently modify previous results. Do not run two jobs into the same output folder.


**Zero-span event update:** supports one-sample fixation/saccade records only when the actual native gaze timestamps verify them; recorded one-sample blinks remain supported. Fixations under100ms stay excluded from sentiment trials. Completed readers from the preceding pinned versions are checksum-verified and skipped; earlier missing-source exclusions are preserved. An unfinished reader restarts CPU preparation. Do not delete saved data.


**Source-file update:** raw and corrected eye-event files are resolved by exact reader/block/type, allowing SR/SNR naming variants. Blocks missing required files are explicitly excluded and reported; corrected eye data is never substituted for the raw event timeline. The update preserves verified CPU checkpoints from the earlier staged versions. Expect `CHECKPOINT MIGRATION` and `KEEP CPU READER` for saved readers. Do not delete the Drive output.


**Blink-record fix:** this version accepts a source-recorded one-sample blink only when its duration matches the measured eye sampling period. The exact previous staged version can be updated while preserving checksum-verified prepared readers such as ZAB. Expect `CHECKPOINT MIGRATION` and `KEEP CPU READER ... ZAB`. An unfinished ZDM reader restarts CPU preparation. No saved reader data is deleted.


## Connect Drive and load the pinned code
Choose the same account for Drive mount and quota authentication. No GitHub token is required.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_recovered'
REPO=Path('/content/word-sentiment-eeg-paper-recovered')
CODE_REVISION='verified-clock-recovery-v1'
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    git('-C',REPO,'checkout','--detach',CODE_REVISION)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## Prepare and save readers

In [ ]:
from datetime import datetime, timezone
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_paper_recovered_stages.py'),
         '--thesis-root',str(THESIS),'--phase','prepare',
         '--budget-bytes','4000000000','--free-drive-bytes',str(FREE_DRIVE_BYTES)]
attempt=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_with_progress(command,OUT/'status'/f'prepare_{attempt}.json',activity='ZuCo paper prepare: persistent CPU to GPU handoff')


## Verify that changing runtimes is safe

In [ ]:
state=json.loads((OUT/'cpu_preparation_manifest.json').read_text())
assert state['status']=='completed'
rows=[]
for subject in state['subjects']:
    item=json.loads((OUT/'readers'/subject/'prepared.json').read_text())
    blocks=json.loads((OUT/'readers'/subject/'source_mapping.json').read_text())['blocks']
    rows.append(dict(reader=subject,negative=item['accepted_counts'][0],positive=item['accepted_counts'][1],pairs=item['pairs'],
                     included_blocks=sum(b['status']=='included' for b in blocks),
                     missing_file_blocks=sum(b['status']=='excluded_missing_source_files' for b in blocks),
                     clock_excluded_blocks=sum(b['status']=='excluded_clock_alignment' for b in blocks)))
display(pd.DataFrame(rows))
print('CPU PREPARATION COMPLETE. You may now switch to GPU.')
print(f'Saved workflow size: {sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())/1e9:.3f} GB')


[Open stage 2: GPU paper analysis](https://colab.research.google.com/github/parmisbathayan/word-sentiment-eeg/blob/main/notebooks/zuco_paper_recover_analyze_gpu.ipynb)